# ❤️ Heart Disease Prediction — Interactive Exploration

This notebook provides an interactive walkthrough of the ML pipeline:
1. Data Loading & Exploration
2. EDA & Visualizations
3. Preprocessing
4. Model Training & Comparison
5. Predictions

In [ ]:
# Import libraries
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Libraries loaded successfully!')

## 1. Load the Dataset

In [ ]:
from src.data_loader import load_heart_data, get_feature_info

df = load_heart_data()
print(f'Dataset shape: {df.shape}')
df.head(10)

In [ ]:
# Feature descriptions
feature_info = get_feature_info()
for name, info in feature_info.items():
    print(f'{name:12s} | {info["type"]:12s} | {info["description"]}')

In [ ]:
# Basic statistics
df.describe().round(2)

## 2. Exploratory Data Analysis

In [ ]:
# Target distribution
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#2ecc71', '#e74c3c']

df['target'].value_counts().plot(kind='bar', ax=ax[0], color=colors, edgecolor='black')
ax[0].set_title('Target Distribution', fontweight='bold')
ax[0].set_xticklabels(['No Disease (0)', 'Disease (1)'], rotation=0)

df['target'].value_counts().plot(kind='pie', ax=ax[1], colors=colors,
    autopct='%1.1f%%', labels=['No Disease', 'Disease'])
ax[1].set_title('Target Proportion', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(df.corr(), dtype=bool))
sns.heatmap(df.corr(), mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Age distribution by target
plt.figure(figsize=(10, 6))
for target, color, label in [(0, '#2ecc71', 'No Disease'), (1, '#e74c3c', 'Disease')]:
    subset = df[df['target'] == target]['age']
    plt.hist(subset, bins=20, alpha=0.6, color=color, label=label, edgecolor='black')
plt.title('Age Distribution by Heart Disease Status', fontweight='bold')
plt.xlabel('Age')
plt.ylabel('Count')
plt.legend()
plt.show()

## 3. Preprocessing

In [ ]:
from src.preprocessing import preprocess_data

X_train, X_test, y_train, y_test, scaler, feature_names = preprocess_data(df)
print(f'\nTraining features shape: {X_train.shape}')
print(f'Testing features shape:  {X_test.shape}')
print(f'Features: {feature_names}')

## 4. Model Training & Evaluation

In [ ]:
from src.models import train_and_evaluate_all

results = train_and_evaluate_all(X_train, X_test, y_train, y_test, tune=True)

In [ ]:
# Results summary table
summary = []
for name, metrics in results.items():
    summary.append({
        'Model': name,
        'Accuracy': f"{metrics['accuracy']:.4f}",
        'Precision': f"{metrics['precision']:.4f}",
        'Recall': f"{metrics['recall']:.4f}",
        'F1-Score': f"{metrics['f1_score']:.4f}",
        'ROC-AUC': f"{metrics['roc_auc']:.4f}" if metrics['roc_auc'] else 'N/A',
    })

pd.DataFrame(summary).set_index('Model')

## 5. Visualize Results

In [ ]:
from src.visualizations import plot_model_comparison, plot_roc_curves, plot_feature_importance

plot_model_comparison(results, save=False)
plt.show()

In [ ]:
plot_roc_curves(results, y_test, save=False)
plt.show()

In [ ]:
plot_feature_importance(results, feature_names, save=False)
plt.show()

## 6. Make a Prediction

In [ ]:
from src.models import predict_single

# Example patient data
patient = {
    'age': 55, 'sex': 1, 'cp': 2, 'trestbps': 140, 'chol': 250,
    'fbs': 0, 'restecg': 1, 'thalach': 150, 'exang': 0,
    'oldpeak': 1.5, 'slope': 1, 'ca': 0, 'thal': 2,
    'age_group': 2, 'hr_reserve': 15, 'chol_level': 2, 'bp_category': 1,
}

best_name = max(results, key=lambda k: results[k]['accuracy'])
best_model = results[best_name]['model']

result = predict_single(best_model, scaler, patient)

print(f"Patient Age: {patient['age']}")
print(f"Diagnosis:   {result['diagnosis']}")
print(f"Confidence:  {result['confidence']:.2%}")